# Q-Mini Add/Sub — Primary FineWiki English assistant training (v2 stability recipe)

This notebook prepares and trains the **primary new Add/Sub model line**. The legacy Q-Mini Light Q2-v2 line is compatibility-only and is not the default training route. This notebook does not load or modify legacy weights, checkpoints, tokenizer or datasets.

The pinned English data mix is a hash-verified 1% FineWiki/Wikipedia sample for factual language pretraining, plus reviewed non-synthetic OpenAssistant (OASST1) conversations and human-authored Dolly instruction examples for assistant behavior. Article IDs and conversation groups are split before training. Attribution/share-alike terms must be reviewed before enabling downloads.

All destructive or expensive actions are **OFF by default**. CPU is the safe default; the notebook automatically selects a conservative CPU thread count. It checks out the PR branch containing the resume-safety and anti-collapse checks. Outputs go into a **new isolated Drive folder** named `trainability_finewiki_v1`; the older `trainability_v2` data and checkpoints are left untouched.

Training still uses floating-point PyTorch optimization. The 5-symbol weight alphabet requires 3-bit storage, and the complete multiplication-free mobile inference runtime remains unfinished. A green CI run is not proof of assistant quality: real held-out testing and repeated-answer checks are mandatory.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.environ['CUDA_VISIBLE_DEVICES'] = ''

import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/lion36508/Q-mini-light-0.7M.git'
REPO_BRANCH = 'fix/dataset-switch-training-reset'
REPO_DIR = Path('/content/Q-mini-light-0.7M')

if not (REPO_DIR / '.git').is_dir():
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', 'fetch', 'origin', REPO_BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'checkout', '-B', REPO_BRANCH, f'origin/{REPO_BRANCH}'], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'pull', '--ff-only', 'origin', REPO_BRANCH], cwd=REPO_DIR, check=True)

DRIVE_ROOT = Path('/content/drive/MyDrive/Q mini 2 light')
VERSION_ROOT = DRIVE_ROOT / 'trainability_finewiki_v1'
DATA_DIR = VERSION_ROOT / 'data'
CACHE_DIR = VERSION_ROOT / 'cache'
USER_DATASET_DIR = VERSION_ROOT / 'selected_dataset'
ARTIFACT_DIR = VERSION_ROOT / 'artifacts'
PRETRAIN_DIR = VERSION_ROOT / 'runs' / 'pretrain'
SFT_DIR = VERSION_ROOT / 'runs' / 'sft'
for folder in (DATA_DIR, CACHE_DIR, ARTIFACT_DIR, PRETRAIN_DIR, SFT_DIR, USER_DATASET_DIR):
    folder.mkdir(parents=True, exist_ok=True)

# Install only required data tools; keep Colab's preinstalled PyTorch intact.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'numpy>=1.24', 'tokenizers>=0.15', 'huggingface_hub>=0.28', 'pyarrow>=15.0', 'zstandard>=0.22'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps', '-q', '-e', str(REPO_DIR)], check=True)

import torch
CPU_THREADS = max(1, min(4, os.cpu_count() or 2))
torch.set_num_threads(CPU_THREADS)
print('PyTorch:', torch.__version__)
print('Available CUDA:', torch.cuda.is_available())
print('CPU threads:', torch.get_num_threads())
print('New model Drive root:', DRIVE_ROOT)


## Source-license review before download

The pinned dataset sources have different reuse terms: FineWiki/Wikipedia sample (**CC-BY-SA-4.0**), OASST1 (**Apache-2.0**), and Dolly 15k (**CC-BY-SA-3.0**). Wikipedia/Dolly attribution and share-alike requirements can affect redistribution of a trained model or derived dataset. Read the source dataset cards and obtain any needed legal review before setting `ACCEPT_SOURCE_LICENSES=True`. This notebook does not provide legal advice.

The complete English FineWiki corpus is far larger than this CPU-friendly 1% sample. The sample is not the entire Wikipedia dump and is not represented as a complete knowledge base.

In [ ]:
# Select ONE data path only after checking the dataset and its license.
PREPARE_PINNED_CORPUS = False
IMPORT_USER_DATA = False
ACCEPT_SOURCE_LICENSES = False

# For a user-selected dataset, place these files in the Drive folder printed above.
USER_PRETRAIN_TRAIN = USER_DATASET_DIR / 'train.txt'
USER_PRETRAIN_VALIDATION = USER_DATASET_DIR / 'validation.txt'
USER_PRETRAIN_TEST = USER_DATASET_DIR / 'test.txt'
USER_PRETRAIN_SOURCE = ''   # Dataset name / revision / source URL.
USER_PRETRAIN_LICENSE = ''  # Exact license shown by the source.

# Optional supervised assistant data. Leave both paths as None until selected.
USER_SFT_TRAIN = None
USER_SFT_VALIDATION = None
USER_SFT_TEST = None
USER_SFT_SOURCE = ''
USER_SFT_LICENSE = ''

if PREPARE_PINNED_CORPUS and IMPORT_USER_DATA:
    raise ValueError('Choose either the pinned corpus or your own local dataset, not both.')

if PREPARE_PINNED_CORPUS:
    if not ACCEPT_SOURCE_LICENSES:
        raise RuntimeError('Review FineWiki CC-BY-SA-4.0, OASST1 Apache-2.0 and Dolly CC-BY-SA-3.0 terms, then set ACCEPT_SOURCE_LICENSES=True.')
    prepare_command = [
        sys.executable, 'scripts/prepare_qmini_addsub_data.py',
        '--output', str(DATA_DIR),
        '--cache', str(CACHE_DIR),
        '--accept-source-licenses',
    ]
    subprocess.run(prepare_command, cwd=REPO_DIR, check=True)
    print('Pinned FineWiki/Wikipedia + OASST1 + Dolly English dataset prepared and hashed. No model training has started.')

elif IMPORT_USER_DATA:
    required = [USER_PRETRAIN_TRAIN, USER_PRETRAIN_VALIDATION, USER_PRETRAIN_TEST]
    missing = [str(path) for path in required if not Path(path).is_file()]
    if missing:
        raise FileNotFoundError('Copy the selected train/validation/test text files into Drive first: ' + ', '.join(missing))
    if not USER_PRETRAIN_SOURCE.strip() or not USER_PRETRAIN_LICENSE.strip():
        raise ValueError('Enter the selected dataset source/revision and its license.')
    import_command = [
        sys.executable, 'scripts/import_qmini_addsub_dataset.py',
        '--pretrain-train', str(USER_PRETRAIN_TRAIN),
        '--pretrain-validation', str(USER_PRETRAIN_VALIDATION),
        '--pretrain-test', str(USER_PRETRAIN_TEST),
        '--pretrain-source', USER_PRETRAIN_SOURCE,
        '--pretrain-license', USER_PRETRAIN_LICENSE,
        '--output', str(DATA_DIR),
        '--confirm-rights-and-human-data',
        '--confirm-english-data',
    ]
    if any(value is not None for value in (USER_SFT_TRAIN, USER_SFT_VALIDATION, USER_SFT_TEST)):
        if any(value is None for value in (USER_SFT_TRAIN, USER_SFT_VALIDATION, USER_SFT_TEST)):
            raise ValueError('Provide SFT train, validation and test files together.')
        if not USER_SFT_SOURCE.strip() or not USER_SFT_LICENSE.strip():
            raise ValueError('Enter the SFT dataset source and license.')
        import_command += [
            '--sft-train', str(USER_SFT_TRAIN),
            '--sft-validation', str(USER_SFT_VALIDATION),
            '--sft-test', str(USER_SFT_TEST),
            '--sft-source', USER_SFT_SOURCE,
            '--sft-license', USER_SFT_LICENSE,
        ]
    subprocess.run(import_command, cwd=REPO_DIR, check=True)
    print('Your selected local English data has been imported. No dataset was downloaded.')

else:
    print('DATA PREPARATION IS OFF. No dataset has been downloaded or imported.')
    print('After choosing a dataset, set IMPORT_USER_DATA=True and fill the Drive paths/source/license above.')


In [ ]:
BUILD_TOKENIZER_AND_STREAMS = False
TOKENIZER_PATH = ARTIFACT_DIR / 'tokenizer.json'
MODEL_CONFIG_PATH = ARTIFACT_DIR / 'model_config.json'

if BUILD_TOKENIZER_AND_STREAMS:
    if not (DATA_DIR / 'pretrain_train.txt').is_file() or not (DATA_DIR / 'dataset_manifest.json').is_file():
        raise FileNotFoundError('Prepare/import your selected dataset first.')
    subprocess.run([
        sys.executable, 'scripts/train_qmini_addsub_tokenizer.py',
        '--data-dir', str(DATA_DIR),
        '--output', str(TOKENIZER_PATH),
        '--vocab-size', '1024',
    ], cwd=REPO_DIR, check=True)

    subprocess.run([
        sys.executable, 'scripts/encode_qmini_addsub_streams.py',
        '--data-dir', str(DATA_DIR),
        '--tokenizer', str(TOKENIZER_PATH),
    ], cwd=REPO_DIR, check=True)
    print('Tokenizer and verified train/validation/test token streams are ready.')
else:
    print('TOKENIZER BUILD AND STREAM ENCODING ARE OFF.');
    print('Set BUILD_TOKENIZER_AND_STREAMS=True only after reviewing and importing your dataset.')


## Stage 1 — base English pretraining

Start from random initialization. This stage writes checkpoints only under the separate Add/Sub v1 Drive folder. The default is CPU. The suggested schedule is a measured 15,000-step first full-data run with 512-token windows; it is not a guarantee of completion time or quality. Check the validation curve and measured token throughput before increasing it.

In [ ]:
START_PRETRAIN = False  # Change to True only when ready to begin a new model training run.
DEVICE = 'cpu'          # Safe default. Set to 'cuda' only when you intentionally use a GPU.
PRETRAIN_STEPS = 15000
BATCH_SIZE = 2 if DEVICE == 'cpu' else 8
SEQUENCE_LENGTH = 512
PRETRAIN_RESUME = None  # Set to a checkpoint from this Add/Sub v1 run only.

if START_PRETRAIN:
    required_files = [DATA_DIR / 'pretrain_train.u16', DATA_DIR / 'pretrain_validation.u16', TOKENIZER_PATH, MODEL_CONFIG_PATH, DATA_DIR / 'dataset_manifest.json']
    missing = [str(path) for path in required_files if not Path(path).is_file()]
    if missing:
        raise FileNotFoundError('Pretraining cannot start; missing selected dataset/tokenizer artifacts: ' + ', '.join(missing))
    command = [
        sys.executable, '-m', 'qmini_addsub.train',
        '--train-bin', str(DATA_DIR / 'pretrain_train.u16'),
        '--valid-bin', str(DATA_DIR / 'pretrain_validation.u16'),
        '--tokenizer', str(TOKENIZER_PATH),
        '--config', str(MODEL_CONFIG_PATH),
        '--output', str(PRETRAIN_DIR),
        '--device', DEVICE,
        '--steps', str(PRETRAIN_STEPS),
        '--batch-size', str(BATCH_SIZE),
        '--sequence-length', str(SEQUENCE_LENGTH),
        '--threads', str(CPU_THREADS),
    ]
    if PRETRAIN_RESUME:
        command += ['--resume', str(PRETRAIN_RESUME)]
    subprocess.run(command, cwd=REPO_DIR, check=True)
else:
    print('Pretraining is OFF. Set START_PRETRAIN=True to start from random initialization.')


## Stage 2 — supervised instruction fine-tuning

Only run after Stage 1 has produced a non-zero-step Add/Sub checkpoint, validation loss is improving, and the training outputs are not collapsed. The SFT trainer rejects legacy Q2-v2 checkpoints and random step-zero artifacts.

In [ ]:
START_SFT = False
SFT_STEPS = 2000
SFT_SAVE_EVERY = 100
SFT_DEVICE = DEVICE
PRETRAIN_CHECKPOINT = None  # Set to an Add/Sub v1 pretraining checkpoint path.
SFT_RESUME = None            # Optional Add/Sub v1 SFT checkpoint only.

if START_SFT:
    required_sft_files = [TOKENIZER_PATH, DATA_DIR / 'sft_train.jsonl', DATA_DIR / 'sft_validation.jsonl', DATA_DIR / 'sft_test.jsonl', DATA_DIR / 'dataset_manifest.json']
    missing = [str(path) for path in required_sft_files if not Path(path).is_file()]
    if missing:
        raise FileNotFoundError('SFT cannot start; import reviewed human-authored SFT splits first: ' + ', '.join(missing))
    if PRETRAIN_CHECKPOINT is None:
        candidates = sorted(PRETRAIN_DIR.glob('checkpoint-step-*.pt'))
        if not candidates:
            raise FileNotFoundError('No Add/Sub v1 pretraining checkpoint found. Finish Stage 1 first.')
        PRETRAIN_CHECKPOINT = candidates[-1]
    command = [
        sys.executable, '-m', 'qmini_addsub.sft',
        '--base-checkpoint', str(PRETRAIN_CHECKPOINT),
        '--tokenizer', str(TOKENIZER_PATH),
        '--train-jsonl', str(DATA_DIR / 'sft_train.jsonl'),
        '--valid-jsonl', str(DATA_DIR / 'sft_validation.jsonl'),
        '--test-jsonl', str(DATA_DIR / 'sft_test.jsonl'),
        '--output', str(SFT_DIR),
        '--device', SFT_DEVICE,
        '--steps', str(SFT_STEPS),
        '--save-every', str(SFT_SAVE_EVERY),
        '--threads', str(CPU_THREADS),
    ]
    if SFT_RESUME:
        command += ['--resume', str(SFT_RESUME)]
    subprocess.run(command, cwd=REPO_DIR, check=True)
else:
    print('SFT is OFF. Enable only after checking the new model pretraining checkpoint.')


In [ ]:
print('Add/Sub v1 data:', DATA_DIR)
print('Add/Sub v1 tokenizer:', TOKENIZER_PATH)
print('Pretraining checkpoints:', PRETRAIN_DIR)
print('SFT checkpoints:', SFT_DIR)
print('Legacy model and legacy Drive folder were not selected by this notebook.')
print('Full integer-only mobile inference is not yet complete.')

RUN_HELDOUT_EVAL = False
HELDOUT_EVAL_CHECKPOINT = None
HELDOUT_EVAL_DIR = VERSION_ROOT / 'evaluation' / 'heldout'
if RUN_HELDOUT_EVAL:
    required = [TOKENIZER_PATH, DATA_DIR / 'sft_test.jsonl', DATA_DIR / 'dataset_manifest.json']
    missing = [str(path) for path in required if not Path(path).is_file()]
    if missing:
        raise FileNotFoundError('Held-out SFT evaluation cannot start; missing: ' + ', '.join(missing))
    if HELDOUT_EVAL_CHECKPOINT is None:
        candidates = sorted(SFT_DIR.glob('sft-checkpoint-step-*.pt'))
        if not candidates:
            raise FileNotFoundError('No SFT checkpoint exists for assistant evaluation.')
        HELDOUT_EVAL_CHECKPOINT = candidates[-1]
    subprocess.run([
        sys.executable, '-m', 'qmini_addsub.evaluate',
        '--checkpoint', str(HELDOUT_EVAL_CHECKPOINT),
        '--tokenizer', str(TOKENIZER_PATH),
        '--test-jsonl', str(DATA_DIR / 'sft_test.jsonl'),
        '--dataset-manifest', str(DATA_DIR / 'dataset_manifest.json'),
        '--output', str(HELDOUT_EVAL_DIR),
        '--device', DEVICE,
    ], cwd=REPO_DIR, check=True)
else:
    print('Held-out assistant evaluation is OFF. Set RUN_HELDOUT_EVAL=True after SFT.')

EXPORT_PACKED_WEIGHTS = False
EXPORT_CHECKPOINT = None  # Optional explicit Add/Sub pretraining or SFT checkpoint.
EXPORT_PATH = ARTIFACT_DIR / 'qmini_addsub_model.qma'

if EXPORT_PACKED_WEIGHTS:
    if EXPORT_CHECKPOINT is None:
        candidates = sorted(SFT_DIR.glob('sft-checkpoint-step-*.pt'))
        if not candidates:
            candidates = sorted(PRETRAIN_DIR.glob('checkpoint-step-*.pt'))
        if not candidates:
            raise FileNotFoundError('No trained Add/Sub checkpoint is available to export.')
        EXPORT_CHECKPOINT = candidates[-1]
    subprocess.run([
        sys.executable, '-m', 'qmini_addsub.export',
        '--checkpoint', str(EXPORT_CHECKPOINT),
        '--output', str(EXPORT_PATH),
    ], cwd=REPO_DIR, check=True)
else:
    print('Packed weight export is OFF. Set EXPORT_PACKED_WEIGHTS=True after training.')
